# NOMA Deep Learning Model Training

This notebook implements the training and evaluation pipeline for the NOMA deep learning model using Google Colab. The implementation is organized in a modular way for better maintainability and debugging.

## 1. Setup Google Drive and Environment

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set project directory
import os
PROJECT_DIR = '/content/drive/My Drive/NOMA_Project'
os.chdir(PROJECT_DIR)

# Install required packages
!pip install -r requirements.txt

## 2. Import Dependencies

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Import custom modules
from models.noma_net import NOMANet
from utils.data_loader import NOMADataset
from utils.metrics import calculate_metrics
from configs.config import load_config

## 3. Configuration

In [ ]:
# Load configuration
config = load_config('configs/training_config.yaml')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 4. Data Loading and Preprocessing

In [ ]:
# Initialize datasets
train_dataset = NOMADataset(
    data_dir=config['data_dir'],
    split='train',
    transform=config['transforms']
)

val_dataset = NOMADataset(
    data_dir=config['data_dir'],
    split='val',
    transform=config['transforms']
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=2
)

## 5. Model Definition

In [ ]:
# Initialize model
model = NOMANet(
    input_dim=config['input_dim'],
    hidden_dim=config['hidden_dim'],
    output_dim=config['output_dim']
).to(device)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=0.1, 
    patience=5
)

## 6. Training Function

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    
    for batch_idx, (data, target) in enumerate(tqdm(train_loader)):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            total_loss += loss.item()
    
    return total_loss / len(val_loader)

## 7. Training Loop

In [ ]:
# Training loop
num_epochs = config['num_epochs']
best_val_loss = float('inf')
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    
    # Validate
    val_loss = validate(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    
    # Update learning rate
    scheduler.step(val_loss)
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f'{config["save_dir"]}/best_model.pth')
    
    print(f'Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}')

## 8. Visualization

In [ ]:
# Plot training curves
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.savefig(f'{config["save_dir"]}/training_curves.png')
plt.show()

## 9. Model Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load(f'{config["save_dir"]}/best_model.pth'))
model.eval()

# Calculate and print metrics
metrics = calculate_metrics(model, val_loader, device)
for metric_name, value in metrics.items():
    print(f'{metric_name}: {value:.4f}')